# Deep learning models
In the other tutorials, all examples have used scikit-learn models. However,
QSPRpred also has a number of other deep-learning models build-in. These models rely on
torch, therefore you need to make sure to have torch or installed QSPPred with the `deep` (or `full`) option (see [README.txt](https://github.com/CDDLeiden/QSPRpred#readme)).

First, we will load the dataset as usual.

In [1]:
import os
import sys
sys.path.insert(0, '/home/ubuntu/Bakalarka/QSPRpred/qsprpred/extra/gpu/models')
from MLP import MultiLayerPerceptron
sys.path.insert(0, '/home/ubuntu/Bakalarka/QSPRpred/tutorials/advanced/data/')
from qsprpred.data import QSPRDataset, RandomSplit
from qsprpred.data.descriptors.fingerprints import MorganFP


os.makedirs("../../tutorial_output/data", exist_ok=True)

# Create dataset
dataset = QSPRDataset.fromTableFile(
    filename="../../tutorial_data/A2A_LIGANDS.tsv",
    store_dir="../../tutorial_output/data",
    name="DeepLearningTutorialDataset",
    target_props=[{"name": "pchembl_value_Mean", "task": "SINGLECLASS", "th": [6.5]}],
    random_state=42
)

# calculate compound features and split dataset into train and test
dataset.prepareDataset(
    split=RandomSplit(test_fraction=0.2, dataset=dataset),
    feature_calculators=[MorganFP(radius=3, nBits=2048)],
    recalculate_features=True,
)

dataset.getDF().head()

,SMILES,pchembl_value_Mean,Year,QSPRID,pchembl_value_Mean_original
QSPRID,,,,,
DeepLearningTutorialDataset_0000,Cc1cc(C)n(-c2cc(NC(=O)CCN(C)C)nc(-c3ccc(C)o3)n...,True,2008.0,DeepLearningTutorialDataset_0000,8.68
DeepLearningTutorialDataset_0001,Nc1c(C(=O)Nc2ccc([N+](=O)[O-])cc2)sc2nc3c(cc12...,False,2010.0,DeepLearningTutorialDataset_0001,4.82
DeepLearningTutorialDataset_0002,O=C(Nc1nc2ncccc2n2c(=O)n(-c3ccccc3)nc12)c1ccccc1,False,2009.0,DeepLearningTutorialDataset_0002,5.65
DeepLearningTutorialDataset_0003,CNC(=O)C12CC1C(n1cnc3c(NCc4cccc(Cl)c4)nc(C#CCC...,False,2009.0,DeepLearningTutorialDataset_0003,5.45
DeepLearningTutorialDataset_0004,CCCn1c(=O)c2c(nc3cc(OC)ccn32)n(CCCNC(=O)c2ccc(...,False,2019.0,DeepLearningTutorialDataset_0004,5.20


In [2]:
print(f"Number of samples train set: {len(dataset.y)}")
print(f"Number of active samples train set: {dataset.y.sum()}")
print(f"Number of samples test set: {len(dataset.y_ind)}")
print(f"Number of active samples test set: {dataset.y_ind.sum()}")

Number of samples train set: 3265
Number of active samples train set: pchembl_value_Mean    2002
dtype: int64
Number of samples test set: 817
Number of active samples test set: pchembl_value_Mean    493
dtype: int64


## Fully connected neural network
### Initialization
The first model we will look at is a fully connected neural network. This model uses the `DDNModel` class instead of the `SklearnModel` class. The `DDNModel` class accepts a `patience` argument, which is the number of epochs to wait before stopping training if the validation loss does not improve and a tolerance ( `tol`) argument, which is the minimum improvement in validation loss to be considered an improvement.

Other parameters for the underlying estimator `STFullyConnected` can be passed to the `parameters` argument as usual.
There is no need to specify the `alg` argument, as currently only `STFullyConnected` is available.

In [3]:
sys.path.insert(0, '/home/ubuntu/Bakalarka/QSPRpred')

from qsprpred.extra.gpu.models.dnn import DNNModel


os.makedirs("../../tutorial_output/models", exist_ok=True)
model = DNNModel(
    base_dir='../../tutorial_output/models',
    name='DeepLearningTutorialModel',
    parameters={"is_reg":False, "print_outputs": 0},
    patience=3,
    tol=0.1,
    random_state=11,
    autoload=False,
    alg=MultiLayerPerceptron
    
)
print(model.__dict__)
print("\nModel object created successfully with local code.")

{'device': device(type='cuda', index=0), 'gpus': (0,), 'patience': 3, 'tol': 0.1, 'nClass': None, 'nDim': None, 'name': 'DeepLearningTutorialModel', 'baseDir': '/home/ubuntu/Bakalarka/QSPRpred/tutorials/tutorial_output/models', 'targetProperties': None, 'nTargets': None, 'featureCalculators': None, 'featureStandardizer': None, 'earlyStopping': <qsprpred.models.early_stopping.EarlyStopping object at 0x7faecda12890>, 'parameters': {'is_reg': False, 'print_outputs': 0, 'seed': 11}, 'alg': <class 'MLP.MultiLayerPerceptron'>, 'randomState': 11, 'estimator': 'Uninitialized model.'}

Model object created successfully with local code.


In [4]:
from qsprpred.models import CrossValAssessor, TestSetAssessor
from qsprpred.models import EarlyStoppingMode

CrossValAssessor("accuracy", mode=EarlyStoppingMode.RECORDING)(model, dataset)
model.earlyStopping.trainedEpochs

[1, 1, 4, 1, 1]

In [5]:
TestSetAssessor("accuracy")(model, dataset)


array([0.74296206])

In [6]:
model.fitDataset(dataset)

'/home/ubuntu/Bakalarka/QSPRpred/tutorials/tutorial_output/models/DeepLearningTutorialModel/DeepLearningTutorialModel_meta.json'

In [7]:
model.estimator.get_params()

{'act_fun': <function torch.nn.functional.relu(input: torch.Tensor, inplace: bool = False) -> torch.Tensor>,
 'batch_size': 256,
 'device': device(type='cuda', index=0),
 'dropout_frac': 0.25,
 'gpus': (0,),
 'is_reg': False,
 'lr': 0.0001,
 'n_class': 2,
 'n_dim': 2048,
 'n_epochs': 2,
 'neuron_layers': [2048, 1024],
 'optimizer': torch.optim.adamw.AdamW,
 'patience': 3,
 'print_outputs': 0,
 'seed': 11,
 'tol': 0.1,
 'weight_decay': 0}

In [8]:
TestSetAssessor("accuracy")(model, dataset)

array([0.77356181])

### Training
Below we will show a complete training of the DNN. First we run hyperparameter optimization to find the best parameters for the model. Here we will will use `EarlyStoppingMode.NOT_RECORDING` as the best epoch to stop training may depend on the hyper-parameters. Then we will apply cross-validation and test set evaluation to get an estimate of the performance of the model, with early stopping set to `EarlyStoppingMode.RECORDING`. Finally, we will use `QSPRModel.fitDataset` training for exactly the average number of epochs trained for in the cross-validation and the test set evaluation.

In [9]:
from qsprpred.models import GridSearchOptimization, TestSetAssessor

# Define the search space
search_space = {"lr": [1e-4 ], "neuron_layers": [[1000, 2000],[10], [20]], "n_epochs":[100], "print_outputs": [2]}

gridsearcher = GridSearchOptimization(
    param_grid=search_space,
    model_assessor=TestSetAssessor(
        scoring='accuracy',
        mode=EarlyStoppingMode.NOT_RECORDING
    ),
)
gridsearcher.optimize(model, dataset, refit_optimal=True)
print(model.estimator.get_params())

# Create a CrossValAssessor object
#CrossValAssessor('roc_auc', mode=EarlyStoppingMode.RECORDING)(model, dataset)
#TestSetAssessor('roc_auc', mode=EarlyStoppingMode.RECORDING)(model, dataset)

#_ = model.fitDataset(dataset, mode=EarlyStoppingMode.OPTIMAL)


Epoch 1 | Train Loss: 0.5287 | Valid Loss: 0.5265 | MCC: 0.4303
Epoch 2 | Train Loss: 0.5124 | Valid Loss: 0.5105 | MCC: 0.4692
Epoch 3 | Train Loss: 0.4863 | Valid Loss: 0.4788 | MCC: 0.5041
Epoch 4 | Train Loss: 0.4390 | Valid Loss: 0.4263 | MCC: 0.5220
Epoch 5 | Train Loss: 0.3760 | Valid Loss: 0.3729 | MCC: 0.5739
Epoch 6 | Train Loss: 0.3264 | Valid Loss: 0.3462 | MCC: 0.5632
Epoch 7 | Train Loss: 0.2818 | Valid Loss: 0.3425 | MCC: 0.5739
Early stopping at epoch 7 | Best Valid Loss: 0.4263
Epoch 1 | Train Loss: 0.5340 | Valid Loss: 0.5368 | MCC: 0.0704
Epoch 2 | Train Loss: 0.5318 | Valid Loss: 0.5350 | MCC: 0.1487
Epoch 3 | Train Loss: 0.5291 | Valid Loss: 0.5325 | MCC: 0.2974
Epoch 4 | Train Loss: 0.5262 | Valid Loss: 0.5292 | MCC: 0.3593
Early stopping at epoch 4 | Best Valid Loss: 0.5368
Epoch 1 | Train Loss: 0.5370 | Valid Loss: 0.5427 | MCC: 0.0000
Epoch 2 | Train Loss: 0.5335 | Valid Loss: 0.5397 | MCC: 0.0000
Epoch 3 | Train Loss: 0.5294 | Valid Loss: 0.5361 | MCC: 0.0673


In [10]:
print(model.estimator.get_params())


{'act_fun': <function relu at 0x7faefa6645e0>, 'batch_size': 256, 'device': device(type='cuda', index=0), 'dropout_frac': 0.25, 'gpus': (0,), 'is_reg': False, 'lr': 0.0001, 'n_class': 2, 'n_dim': 2048, 'n_epochs': 100, 'neuron_layers': [1000, 2000], 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 3, 'print_outputs': 2, 'seed': 11, 'tol': 0.1, 'weight_decay': 0}


In [11]:
type(model)

qsprpred.extra.gpu.models.dnn.DNNModel

In [12]:
results_from_dnn_model = model.predict(dataset.X_ind)

In [13]:
model.predict(dataset.X_ind)

array([[0],
       [0],
       [1],
       [1],
       [1],
       [1],
       [0],
       [0],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [0],
       [1],
       [0],
       [1],
       [1],
       [1],
       [1],
       [0],
       [0],
       [0],
       [0],
       [1],
       [0],
       [1],
       [1],
       [1],
       [1],
       [1],
       [0],
       [0],
       [1],
       [1],
       [1],
       [1],
       [1],
       [0],
       [1],
       [0],
       [1],
       [1],
       [0],
       [1],
       [1],
       [1],
       [1],
       [1],
       [0],
       [1],
       [1],
       [1],
       [1],
       [0],
       [0],
       [1],
       [1],
       [1],
       [1],
       [0],
       [1],
       [1],
       [0],
       [1],
       [0],
       [0],
       [1],
       [0],
       [0],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [1],
       [0],
       [0],
       [1],
       [0],
    

In [14]:
result_with_estimator = model.estimator.predict(dataset.X_ind)

In [15]:
from sklearn.metrics import accuracy_score
accuracy_score(dataset.y_ind, result_with_estimator[:,1:]> 0.5)

0.7307221542227662

In [16]:
accuracy_score(dataset.y_ind, results_from_dnn_model)

0.7307221542227662

In [17]:
result_with_estimator == results_from_dnn_model

array([[False, False],
       [False, False],
       [False, False],
       ...,
       [False, False],
       [False, False],
       [False, False]])